In [13]:
import os, glob, shutil, requests, json, calendar, locale
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# deutsche Datumsbezeichnungen verwenden
locale.setlocale(locale.LC_TIME, "de_DE") 

#Server-Key für Trello angeben (MUSS VOM KUNDEN ANGEPASST WERDEN)
key = "cf96243c3494a8473e5caccd2d0f906c"

#Token für Trello angeben (MUSS VOM KUNDEN ANGEPASST WERDEN)
token = "11cb90ee1fbb4081bb760fdba3f70fbb6112177df1921e8d2215d0e1644a7228"

#Board-ID des Produktionsboard eingeben, in dem die Produktionskarten angelegt werden sollen (MUSS VOM KUNDEN ANGEPASST WERDEN)
produktion_board_id = 'bE8Wi83G'

#Board-ID des Logistikboard eingeben, in dem die Logistikkarten angelegt werden sollen (MUSS VOM KUNDEN ANGEPASST WERDEN)
logistik_board_id = '9WeW7Q9E'

#variable Liste mit Bezeichnungen der aktuellen Maschinen, die bei HPN im Einsatz sind 
#Bezeichnungen müssen mit Listen-Bezeichnungen in Produktionsboard übereinstimmen
#(MUSS VOM KUNDEN BEI NEUANLAGE VON MASCHINEN ERGÄNZT ODER BEI AUSSERBETRIEBNAHME GEKÜRZT WERDEN)
alle_produktionslisten = ['StortiX', '21X', 'HAPX', 'M4', 'M5', 'M6']

#Liste mit Bezeichnungen aller Logistiklisten im Logistikboard
#Bezeichnungen müssen mit Listen-Bezeichnungen in Logistikboard übereinstimmen
alle_logistiklisten = ['Montag', 'Dienstag', 'Mittwoch', 'Donnerstag', 'Freitag',
                      'Montag WV', 'Dienstag WV', 'Mittwoch WV', 'Donnerstag WV', 'Freitag WV']

#Bezeichnung der Exportdateien, nach der in den Ordnern gesucht werden soll (MUSS VOM KUNDEN ANGEPASST WERDEN)
search_for = "*export*"

#Bezeichnungen der Spalten, nach denen in Exportdateien zur Erstellung von Produktionskarten gesucht werden soll 
search_list_P = ['Pos_Bezugsquelle', 
                 'F2:Kunde:Name', 
                 'Pos_Bezeichnung',
                 'Pos_Menge', 
                 'Pos_Artikel_Deutsch_Bezeichnung',
                 'Pos_Wunschtermin',
                 'Pos_Artikel_Deutsch_SachmerkmalFeld1']

#Bezeichnungen, nach denen in Exportdateien zur Erstellung von Logistikkarten gesucht werden soll 
search_list_L = ['F2:Auftragsnummer', 
                 'F2:Kunde:Name', 
                 'F2:Versandanschrift4', 
                 'F2:Versandanschrift3',
                 'F2:Versandart',
                 'Pos_Wunschtermin',
                 'Pos_Bezeichnung',
                 'Pos_Bezugsquelle',
                 'Pos_Artikel_Deutsch_Bezeichnung',
                 'Pos_Artikel_Deutsch_SachmerkmalFeld2',
                 'Pos_Menge']

#Pfad, wo sich die Exportdateien befinden (MUSS VOM KUNDEN ANGEPASST WERDEN)
root = 'C:/path/to/your/export/files'os.chdir(root)

###########################
   
#Anlegen von neuen Ordnern
def createFolder(directory):
    try:
        if not os.path.exists(directory):
            os.makedirs(directory)
    except OSError:
        print ('Error: Creating directory. ' + directory)
            
#Zwei Ordner für erledigte und noch nicht erledigte Dateien anlegen
createFolder(root + './Erledigt/')
createFolder(root +'./Noch zu erledigen/')

#Directory, in dem sich noch zu erledigende Dateien befinden sollen
dst_dirname = root + '/Noch zu erledigen/'

#Klasse mit Funktionen zum Hochladen von Daten in Trello
class trello:

    #Erstellen von neuen Karten in Trello-Liste   
    def create_card(self, card_name, list_id):
        
        querystring = {
            "name": card_name, 
            "idList": list_id, 
            "key": key, 
            "token": token
        }

        response = requests.request(
            "POST", 
            url = f"https://api.trello.com/1/cards", 
            params=querystring
        )

        card_id = response.json()["id"]
        
        return card_id

    #Bestimmen der List-ID eines Boards anhand der Listen-Bezeichnung
    def get_list_id(self, board_id, list_name):

        query = {
           'key': key,
           'token': token
        }

        response = requests.request(
           "GET",
           url = "https://api.trello.com/1/boards/" + board_id + "/lists",
           params=query
        )

        list_id_index = [*range(len(json.loads(response.text)))]

        keys = []
        for i in list_id_index:
            keys.append(json.loads(response.text)[i]['name'])

        values = []
        for i in list_id_index:
            values.append(json.loads(response.text)[i]['id'])

        list_id_dict = {}
        for i in list_id_index:
            list_id_dict[keys[i]] = values[i]

        return list_id_dict[list_name]

    #Bestimmen der Ankerpunkte in einer Exportdatei
    def get_anker_punkte(self, export_data):
        
        ankerpunkte = np.where(pd.notna(export_data.drop([0,1,2])[0]))
        ankerpunkte = np.asarray(ankerpunkte).tolist()[0][0:]
        ankerpunkte.remove(0)
        
        return ankerpunkte

    #Abprüfen, ob zu einem Suchbegriff in der Exportdatei Daten vorhanden sind und wenn nein, 
    #die Datei in den Ordner 'Noch zu erledigen' kopieren
    def move_excel(self, search, export_data, position):
        
        row = export_data.loc[export_data.isin([search]).any(axis=1)].index[0]
        col = export_data.T.loc[export_data.T.isin([search]).any(axis=1)].index[0] 
        if pd.isnull(export_data.iloc[row+position, col]) is True:
            dst_filename = os.path.join(dst_dirname, os.path.basename(source_file))
            shutil.copy(source_file, dst_filename)

    #Auflisten aller Dateien im aktuellen Verzeichnis, die einen Suchbegriff im Dateinamen enthalten
    def get_files(self, search_term):
        
        liste_file = []
        for file in glob.glob(search_term):
            liste_file.append(file)
            
        return liste_file  
    
    #Auslesen der Infos aus einer Spalte anhand eines Suchbegriffs
    def get_info(self, export, search, liste, position):
        
        row = export.loc[export.isin([search]).any(axis=1)].index[0]
        col = export.T.loc[export.T.isin([search]).any(axis=1)].index[0]
        liste.append(str(export.iloc[row+position, col]))
        
        return liste      

    #Einlesen der Daten, die in den Kopf der Produktionskarte sollen
    def trello_head_P(self, export, ankerpunkte, search):
        
        liste_head = []
        
        #(1) Auftragsnummer Produktion hinzufügen
        liste_auftragsnummerproduktion = []
        for i in ankerpunkte:
            search = 'Pos_Bezugsquelle' 
            row = export.loc[export.isin([search]).any(axis=1)].index[0]
            col = export.T.loc[export.T.isin([search]).any(axis=1)].index[0]
            liste_auftragsnummerproduktion.append(export.iloc[row+i, col])
        
        #Endungen entfernen
        liste_auftragsnummerproduktion = [x.replace("-00-00", "") for x in liste_auftragsnummerproduktion]

        liste_head.append(liste_auftragsnummerproduktion)

        #(2) Kunde hinzufügen
        liste_kunde = []
        for i in ankerpunkte:
            search = 'F2:Kunde:Name' 
            row = export.loc[export.isin([search]).any(axis=1)].index[0]
            col = export.T.loc[export.T.isin([search]).any(axis=1)].index[0]
            liste_kunde.append(export.iloc[row+1, col]) 

        liste_head.append(liste_kunde)

        #(3) Artikelnummer hinzufügen
        liste_artikelnummer = []
        for i in ankerpunkte:
            search = 'Pos_Bezeichnung' 
            row = export.loc[export.isin([search]).any(axis=1)].index[0]
            col = export.T.loc[export.T.isin([search]).any(axis=1)].index[0]
            liste_artikelnummer.append(export.iloc[row+i, col]) 

        liste_head.append(liste_artikelnummer)
        
        #(4) Stückzahl hinzufügen
        liste_stückzahl = []
        for i in ankerpunkte:
            search = 'Pos_Menge' 
            row = export.loc[export.isin([search]).any(axis=1)].index[0]
            col = export.T.loc[export.T.isin([search]).any(axis=1)].index[0]
            liste_stückzahl.append(str(export.loc[row+i,col]) + ' Stück')

        liste_head.append(liste_stückzahl)

        #(5) Beschreibung hinzufügen
        liste_beschreibung = []
        for i in ankerpunkte:
            search = 'Pos_Bezeichnung' 
            row = export.loc[export.isin([search]).any(axis=1)].index[0]
            col = export.T.loc[export.T.isin([search]).any(axis=1)].index[0]
            liste_beschreibung.append(export.iloc[row+i+2, col]) 

        liste_head.append(liste_beschreibung)

        liste_head = np.array(liste_head).T.tolist()

        return liste_head   
    
    #Einlesen der Daten, die in den Body der Produktionskarte sollen
    def trello_body_P(self, export, ankerpunkte, search):
        
        liste_body = []
        
        #(6) Fälligkeitsdatum hinzufügen
        liste_fälligkeitsdatum = []
        for i in ankerpunkte:
            new_object.get_info(export, 'Pos_Wunschtermin', liste_fälligkeitsdatum, i)

        liste_body.append(liste_fälligkeitsdatum)

        liste_body = np.array(liste_body).T.tolist()

        return liste_body
        
    #Einlesen der Daten, die in den Kopf der Logistikkarte sollen
    def trello_head_L(self, export, ankerpunkte, search):
        
        liste_head = []
        
        #(1) Auftragsnummer hinzufügen
        liste_auftragsnummer = []
        for i in ankerpunkte:
            new_object.get_info(export, 'F2:Auftragsnummer', liste_auftragsnummer, 1)

        liste_head.append(liste_auftragsnummer)
            
        #(2) Kunde hinzufügen
        liste_kunde = []
        for i in ankerpunkte:
            new_object.get_info(export, 'F2:Kunde:Name', liste_kunde, 1)

        liste_head.append(liste_kunde)

        #(3) Versandadresse hinzufügen
        liste_versandadresse = []
        for i in ankerpunkte:
            new_object.get_info(export, 'F2:Versandanschrift4', liste_versandadresse, 1)
         
        liste_head.append(liste_versandadresse)
        
        #(4) Erweiterung Versandadresse hinzufügen
        liste_erweiterung_versandadresse = []
        for i in ankerpunkte:
            new_object.get_info(export, 'F2:Versandanschrift3', liste_erweiterung_versandadresse, 1)
        
        liste_head.append(liste_erweiterung_versandadresse)

        #(5) Versandart hinzufügen
        liste_versandart = []
        for i in ankerpunkte:
            new_object.get_info(export, 'F2:Versandart', liste_versandart, 1)

        liste_head.append(liste_versandart)

        liste_head = np.array(liste_head).T.tolist()

        return liste_head       
    
    #Einlesen der Daten, die in die Beschreibung der Logistikkarte sollen
    def trello_footer_L(self, export, ankerpunkte, search):

        #(10) Verladetag hinzufügen
        liste_verladetag = []
        for i in ankerpunkte:
            search = 'Pos_Wunschtermin' 
            row = export.loc[export.isin([search]).any(axis=1)].index[0]
            col = export.T.loc[export.T.isin([search]).any(axis=1)].index[0]
            ts = str(export.iloc[row+i,col])
            dt = datetime.strptime(ts, '%Y-%m-%d %H:%M:%S')
            #Wochentag des Datums bestimmen
            verladetag = calendar.day_name[dt.weekday()]
            liste_verladetag.append(verladetag)
            
        return liste_verladetag
    
    #Hinzufügen von Daten in vorhandene Produktionskarte
    def add_to_production_card(self, P_list_id, head_P, dt, trello_card_id):
        
        #Produktionskarte ohne Beschreibung erzeugen
        headers = {
           "Accept": "application/json"
        }

        query = {
           'key': key,                                   #API-Schlüssel von Trello
           'token': token,                               #API-Token von Trello
           'idList': P_list_id,                          #ID der Produktionsliste, in der die Karte eingefügt werden soll
           'name': head_P,                               #neuer Name der Karte                             
           'due':  dt,                                   #Frist, bis wann Kartenauftrag erledigt sein muss
           'pos': 'bottom'                               #Position, wo neue Karte in Liste erzeugt werden soll
        }

        response = requests.request(
           "PUT",
           url = "https://api.trello.com/1/cards/" + trello_card_id,
           headers=headers,
           params=query
        )

        card_id = response.json()["id"]

        return card_id 
    
    #Hinzufügen von Daten in vorhandene Logistikkarte
    def add_to_logistic_card(self, L_list_id, head_L, dt, trello_card_id):
        
        #Logistikkarte ohne Beschreibung erzeugen
        headers = {
           "Accept": "application/json"
        }

        query = {
           'key': key,                                   #API-Schlüssel von Trello
           'token': token,                               #API-Token von Trello
           'idList': L_list_id,                          #ID der Produktionsliste, in der die Karte eingefügt werden soll
           'name': head_L,                              #neuer Name der Karte 
           'due':  dt,                                  #Frist, bis wann Kartenauftrag erledigt sein muss
           'pos': 'bottom'                              #Position, wo neue Karte in Liste erzeugt werden soll
        }

        response = requests.request(
           "PUT",
           url = "https://api.trello.com/1/cards/" + trello_card_id,
           headers=headers,
           params=query
        )

        card_id = response.json()["id"]

        return card_id
    
    #Daten für Beschreibung der Produktionskarte hochladen
    def add_description_P(self, P_list_id, head_P, beschreibung, dt, trello_card_id):

        #Beschreibung und Frist ergänzen
        headers = {
           "Accept": "application/json"
        }

        query = {
           'key': key,                                   
           'token': token,                               
           'idList': P_list_id,                                   #ID der Produktionsliste, in die die Karte eingefügt werden soll
           'name': head_P,                                        #neuer Name der Karte 
           'desc': beschreibung,                                  #neue Beschreibung der Karte
           'due':  dt - timedelta(hours=6),                       #Frist, bis wann Kartenauftrag erledigt sein muss abzgl. 6 Stunden wegen Zeitverzug
           'pos': 'bottom'                                        #Position, wo neue Karte in Liste erzeugt werden soll
        }

        response = requests.request(
           "PUT",
           url = "https://api.trello.com/1/cards/" + trello_card_id,
           headers=headers,
           params=query
        )

        card_id = response.json()["id"]

        return card_id
    
    #Daten für Beschreibung der Logistikkarte hochladen
    def add_description_L(self, L_list_id, head_L, verladetag, dt, trello_card_id):

        #Verladetag und Frist ergänzen
        headers = {
           "Accept": "application/json"
        }

        query = {
           'key': key,                                   
           'token': token,                               
           'idList': L_list_id,                                  #ID der Logistikliste, in die die Karte eingefügt werden soll
           'name': head_L,                                       #neuer Name der Karte 
           'desc': "§§" + ''.join(verladetag) + "§§",            #neue Beschreibung der Karte
           'due':  dt - timedelta(hours=6),                      #Frist, bis wann Kartenauftrag erledigt sein muss abzgl. 6 Stunden wegen Zeitverzug
           'pos': 'bottom'                                       #Position, wo neue Karte in Liste erzeugt werden soll
        }

        response = requests.request(
           "PUT",
           url = "https://api.trello.com/1/cards/" + trello_card_id,
           headers=headers,
           params=query
        )

        card_id = response.json()["id"]

        return card_id
    
    #Stückliste als Anhang anfügen
    def add_attachment(self, name, attachment, trello_card_id):

        #MIME-Type der Datei (abhängig von Dateityp der hochzulandenden Datei)
        #.xls: application/vnd.ms-excel
        #.xlsx: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet 
        #.txt: text/plain
        mimeType = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'

        #Datei im Working Directory auslesen
        fl = {'file': (name, open(attachment, 'rb'), mimeType)}

        #Stückliste anhängen
        headers = {
           "Accept": "application/json"
        }

        query = {
           'key': key,
           'token': token
        }

        response = requests.request(
           "POST",
           url = "https://api.trello.com/1/cards/" + trello_card_id + "/attachments",
           headers = headers,
           params = query,
           files = fl
        ) 

        return response
    
    #Anlegen einer leeren Checkliste in einer Produktionskarte
    def add_checklist_P(self, trello_card_id):

        #leere Checkliste erzeugen
        query = {
           'key': key,
           'token': token,
           'name': 'Stückliste'    #Überschrift der Checkliste
        }

        response = requests.request(
           "POST",
           url = "https://api.trello.com/1/cards/" + trello_card_id + "/checklists",
           params=query
        )

        checklist_id = response.json()["id"]

        return checklist_id
    
    #Anlegen einer leeren Checkliste in einer Logistikkarte
    def add_checklist_L(self, trello_card_id):

        #leere Checkliste erzeugen
        query = {
           'key': key,
           'token': token,
           'name': 'Abrufe'    #Überschrift der Checkliste
        }

        response = requests.request(
           "POST",
           url = "https://api.trello.com/1/cards/" + trello_card_id + "/checklists",
           params=query
        )

        checklist_id = response.json()["id"]

        return checklist_id
    
    #Anhängen von Checklistenelementen an Checkliste in Produktionskarte
    def add_checklist_elements_P(self, stückliste_index, stückliste, checklist_id):

        for i in stückliste_index:

            #Checklistenelemente zusammenfügen
            checklist_item = stückliste[i]

            #Checklistenelemente in String umwandeln
            checklist_item = list(map(str, checklist_item))

            #Checklisten-Elemente aneinanderketten
            checklist_item = " ".join(checklist_item)

            #Checkliste befüllen
            query = {
               'key': key,
               'token': token,
               'name': checklist_item,
               'pos': 'bottom'
            }

            response = requests.request(
               "POST",
               url = "https://api.trello.com/1/checklists/" + checklist_id + "/checkItems",
               params=query
            )
            
        return response
    
    #Anhängen von Checklistenelementen an Checkliste in Logistikkarte
    def add_checklist_elements_L(self,
                                 verladetag_index,
                                 liste_artikelnummer, 
                                 liste_bezugsquelle, 
                                 liste_artikelbeschreibung, 
                                 liste_lieferziel, 
                                 listofstapel, 
                                 liste_stückzahl, 
                                 listofverladen, 
                                 checklist_id):
        
        for i in verladetag_index:

            #Checklistenelemente zusammenfügen
            checklist_item = list(zip(liste_artikelnummer, liste_bezugsquelle, liste_artikelbeschreibung, 
                                      liste_lieferziel, listofstapel, liste_stückzahl, listofverladen))[i]

            #Checklisten-Elemente aneinanderketten
            checklist_item = " ".join(checklist_item)

            #Checkliste befüllen
            query = {
               'key': key,
               'token': token,
               'name': checklist_item,
               'pos': 'bottom'
            }

            response = requests.request(
               "POST",
               url = "https://api.trello.com/1/checklists/" + checklist_id + "/checkItems",
               params=query
            )

        return response
         
    #Hochladen von Daten in eine Produktionskarte in Abhängigkeit davon, ob es sich um einen 
    #Produktions- oder Logistikauftrag handelt
    def upload_data_P(self, 
                      liste_auftragsnummerproduktion, 
                      versandart, 
                      P_list_id, 
                      head_P, 
                      dt,
                      beschreibung,
                      name,
                      attachment,
                      stückliste_index,
                      stückliste):

        #falls Auftragnummer Produktion mit 'P' beginnt
        if liste_auftragsnummerproduktion[i].startswith('P'):

            #falls Versandart gleich 'Lager' ist
            if versandart == 'Lager':

                #Produktionskarte erstellen und Produktionsdaten in diese hochladen
                trello_card_id = new_object.create_card('Produktionskarte', P_list_id)
                new_object.add_to_production_card(P_list_id, head_P, dt, trello_card_id)
                new_object.add_description_P(P_list_id, head_P, beschreibung, dt, trello_card_id)
                new_object.add_attachment(name, attachment, trello_card_id)
                checklist_id = new_object.add_checklist_P(trello_card_id)
                new_object.add_checklist_elements_P(stückliste_index, stückliste, checklist_id)

            #falls Versandart nicht 'Lager' ist
            else:
                #Produktionskarte erstellen und Produktionsdaten in diese hochladen
                trello_card_id = new_object.create_card('Produktionskarte', P_list_id)
                new_object.add_to_production_card(P_list_id, head_P, dt, trello_card_id)
                new_object.add_description_P(P_list_id, head_P, beschreibung, dt, trello_card_id)
                new_object.add_attachment(name, attachment, trello_card_id)
                checklist_id = new_object.add_checklist_P(trello_card_id)
                new_object.add_checklist_elements_P(stückliste_index, stückliste, checklist_id)


    #Hochladen von Daten in eine Logistikkarte in Abhängigkeit davon, ob es sich um einen 
    #Produktions- oder Logistikauftrag handelt
    def upload_data_L(self, 
                      liste_auftragsnummerproduktion, 
                      versandart, 
                      L_list_id, 
                      head_L, 
                      dt, 
                      verladetag,
                      verladetag_index,
                      liste_artikelnummer, 
                      liste_bezugsquelle, 
                      liste_artikelbeschreibung, 
                      liste_lieferziel, 
                      listofstapel, 
                      liste_stückzahl, 
                      listofverladen):

        #falls Auftragnummer Produktion mit 'P' beginnt
        if liste_auftragsnummerproduktion[i].startswith('P'):

            #falls Versandart nicht 'Lager' ist
            if versandart != 'Lager':

                #Logistikkarte erstellen und Logistikdaten in diese hochladen
                trello_card_id = new_object.create_card('Logistikkarte', L_list_id)
                new_object.add_to_logistic_card(L_list_id, head_L, dt, trello_card_id)
                new_object.add_description_L(L_list_id, head_L, verladetag, dt, trello_card_id)
                checklist_id = new_object.add_checklist_L(trello_card_id)
                new_object.add_checklist_elements_L(verladetag_index,
                                                    liste_artikelnummer,
                                                    liste_bezugsquelle, 
                                                    liste_artikelbeschreibung,
                                                    liste_lieferziel, 
                                                    listofstapel, 
                                                    liste_stückzahl, 
                                                    listofverladen, 
                                                    checklist_id)

        #falls Auftragnummer Produktion nicht mit 'P' beginnt
        else:
            #Logistikkarte erstellen und Logistikdaten in diese hochladen
            trello_card_id = new_object.create_card('Logistikkarte', L_list_id)
            new_object.add_to_logistic_card(L_list_id, head_L, dt, trello_card_id)
            new_object.add_description_L(L_list_id, head_L, verladetag, dt, trello_card_id)
            checklist_id = new_object.add_checklist_L(trello_card_id)
            new_object.add_checklist_elements_L(verladetag_index,
                                                liste_artikelnummer, 
                                                liste_bezugsquelle,
                                                liste_artikelbeschreibung, 
                                                liste_lieferziel, 
                                                listofstapel, 
                                                liste_stückzahl, 
                                                listofverladen, 
                                                checklist_id)         

new_object = trello()
        
############################
    
#alle Export-Dateien im aktuellen Working Directory auflisten
liste_files = new_object.get_files(search_for)

#Indizes der File-Liste bestimmen
file_index = [*range(len(liste_files))]

#Indizes der Produktionslisten bestimmen
produktionslisten_index = [*range(len(alle_produktionslisten))]

#Indizes der Logistiklisten bestimmen
logistiklisten_index = [*range(len(alle_logistiklisten))]

#Liste mit List-IDs der Produktionslisten, in die die Karten hereingeladen werden sollen, erstellen
liste_P_list_id = []
for i in produktionslisten_index:
    P_list_id_maschine = new_object.get_list_id(produktion_board_id, alle_produktionslisten[i])
    liste_P_list_id.append(P_list_id_maschine)

#Liste mit List-IDs der Logistiklisten, in die die Karten hereingeladen werden sollen, erstellen
liste_L_list_id = []
for i in logistiklisten_index:
    L_list_id_maschine = new_object.get_list_id(logistik_board_id, alle_logistiklisten[i])
    liste_L_list_id.append(L_list_id_maschine)

#Dateien mit fehlenden Angaben für Produktionskarten in Ordner 'Noch zu erledigen' kopieren
for index in file_index:
    
    source_file = liste_files[index]
    
    #Export-Datei einlesen
    export = pd.read_excel(source_file, header = None)
    
    #Ankerpunkte bestimmen
    ankerpunkte = new_object.get_anker_punkte(export)
    
    #Wenn Feld unterhalb Suchbegriff leer ist, zugehörige Exportdatei verschieben
    for i in ankerpunkte:
        for search in search_list_P:
            
            if ((search == 'Pos_Bezugsquelle') or (search == 'Pos_Bezeichnung') or (search == 'Pos_Menge') 
                or (search == 'Pos_Artikel_Deutsch_Bezeichnung') or (search == 'Pos_Wunschtermin')
                or (search == 'Pos_Artikel_Deutsch_SachmerkmalFeld1')):
                new_object.move_excel(search, export, i)

            elif (search == 'F2:Kunde:Name'):
                new_object.move_excel(search, export, 1)

#Dateien mit fehlenden Angaben für Logistikkarten in Ordner 'Noch zu erledigen' kopieren
for index in file_index:
    
    source_file = liste_files[index]
    
    #Export-Datei einlesen
    export = pd.read_excel(source_file, header = None)
    
    #Ankerpunkte bestimmen
    ankerpunkte = new_object.get_anker_punkte(export)
    
    #Wenn Feld unterhalb Suchbegriff leer ist, zugehörige Exportdatei verschieben
    for i in ankerpunkte:
        for search in search_list_L:
            
            if ((search == 'F2:Auftragsnummer') or (search == 'F2:Kunde:Name')
                or (search == 'F2:Versandanschrift4') or (search == 'F2:Versandanschrift3')
                or (search == 'F2:Versandart')):
                new_object.move_excel(search, export, 1)
                
            elif ((search == 'Pos_Wunschtermin') or (search == 'Pos_Bezeichnung') or (search == 'Pos_Bezugsquelle') 
                  or (search == 'Pos_Artikel_Deutsch_Bezeichnung') or (search == 'Pos_Artikel_Deutsch_SachmerkmalFeld2')
                  or (search == 'Pos_Menge')):
                new_object.move_excel(search, export, i)

#Working Directory in Ordner 'Noch zu erledigen' ändern
os.chdir(dst_dirname)

#alle noch zu erledigenden Exportdateien auflisten
liste_tobedone = new_object.get_files(search_for)

#dieselben Dateien im Ursprungsordner löschen
os.chdir(root)
for file in liste_tobedone:
    try:
        os.remove(file)
    except IOError:
        pass

#restliche Export-Dateien im aktuellen Working Directory auflisten
liste_files = new_object.get_files(search_for)

#für jede Export-Datei Karten in zugehöriger Liste in Produktions- und Logistikboard anlegen und befüllen
for file in liste_files:
    
    #Export-Datei einlesen
    export = pd.read_excel(file, header = None)
    
    #Ankerpunkte bestimmen
    ankerpunkte = new_object.get_anker_punkte(export)
    
    #Maschinen auflisten
    liste_maschine = []
    for i in ankerpunkte:
        search = 'Pos_Artikel_Deutsch_SachmerkmalFeld1' 
        row = export.loc[export.isin([search]).any(axis=1)].index[0]
        col = export.T.loc[export.T.isin([search]).any(axis=1)].index[0]
        strg = export.iloc[row+i,col].split('*')[5]
        liste_maschine.append(strg)

    #Indizes der Maschinen-Liste bestimmen
    machine_index = [*range(len(liste_maschine))]
    
    #Indizes der Maschinen, die nur einmal vorkommen
    machine_index_unique = [liste_maschine.index(x) for x in set(liste_maschine)]
    
    #Header für Produktionskarte zusammenfügen
    liste_head_P = []
    head_P = new_object.trello_head_P(export, ankerpunkte, search)
    for i in machine_index:
        head = " | ".join(head_P[i])
        liste_head_P.append(head)
    
    #Header für Logistikkarte zusammenfügen
    liste_head_L = []
    head_L = new_object.trello_head_L(export, ankerpunkte, search)
    for i in machine_index:
        head = " ".join(head_L[i])
        liste_head_L.append(head)
    
    #Wunschliefertermine auflisten
    liste_wunschtermin = new_object.trello_body_P(export, ankerpunkte, search)

    #Artikelbezeichnungen bestimmen
    liste_artikelnummer = []
    for i in ankerpunkte:
        new_object.get_info(export, 'Pos_Bezeichnung', liste_artikelnummer, i)  

    #Dateinamen der Stücklisten in Liste speichern
    files = [s + '-PA.xlsx' for s in liste_artikelnummer]
    
    #Daten aus Stücklisten extrahieren
    liste_stückliste = []
    for i in files:
        #Stückliste zusammenfügen
        liste = []
        df2 = pd.read_excel(i, sheet_name = 'PA', header = None)
        #Produktionsmenge in Feld D2 auslesen
        produktionsmenge = df2.iloc[1,3]
        #Anzahl Artikel pro Position aus Zellen E21 bis E24 auslesen 
        anzahl = df2.iloc[20:24,4].tolist()
        #Gesamtmenge jedes Artikels berechnen durch Multiplikation von Produktionsmenge mit jeweiliger Anzahl Artikel
        gesamtmenge = [i * produktionsmenge for i in anzahl]
        #Daten der Stückliste aus Zellen C21 bis H24 auslesen
        df3 = df2.iloc[20:24,2:8]
        #Spalte 'Gesamtmenge' hinzufügen
        df3[8] = gesamtmenge
        #Spalte 'Stück' hinzufügen
        df3[9] = "Stück"
        lst = df3.values.tolist()
        #(9) Stückliste hinzufügen                        
        liste.extend(lst)
        liste_stückliste.append(liste)
        
    #Indizes der Stücklisten-Elemente bestimmen
    stückliste_index = [*range(len(liste_stückliste[0]))]
    
    #Auftragsnummern Produktion auflisten
    liste_auftragsnummerproduktion = []
    for i in ankerpunkte:
        new_object.get_info(export, 'Pos_Bezugsquelle', liste_auftragsnummerproduktion, i)  

    #Versandart bestimmen
    liste_versandart = []
    versandart = new_object.get_info(export, 'F2:Versandart', liste_versandart, 1)[0] 
    
    #reine Stückzahlen auflisten
    liste_reine_stückzahl = []
    for i in ankerpunkte:
        new_object.get_info(export, 'Pos_Menge', liste_reine_stückzahl, i)
        
    #Artikelbezeichnungen auflisten (oder was sagt die Spalte 'Pos_ArtikelBezeichnung3Kunde' aus?)
    liste_artikelbezeichnung = []
    for i in ankerpunkte:
        new_object.get_info(export, 'Pos_ArtikelBezeichnung3Kunde', liste_artikelbezeichnung, i) 

    #Artikel auflisten (oder was sagt der vorletzte Wert in der Spalte 'Pos_Artikel_Deutsch_SachmerkmalFeld1' aus?)
    liste_artikel= []
    for i in ankerpunkte:
        search = 'Pos_Artikel_Deutsch_SachmerkmalFeld1' 
        row = export.loc[export.isin([search]).any(axis=1)].index[0]
        col = export.T.loc[export.T.isin([search]).any(axis=1)].index[0]
        strg = export.iloc[row+i,col].split('*')[4]
        liste_artikel.append(strg)
        
    #Bezugsquelle hinzufügen
    liste_bezugsquelle = []
    for i in ankerpunkte:
        new_object.get_info(export, 'Pos_Bezugsquelle', liste_bezugsquelle, i)  
    
    #Endungen entfernen
    liste_bezugsquelle = [x.replace("-00-00", "") for x in liste_bezugsquelle]

    #Artikelbeschreibung hinzufügen
    liste_artikelbeschreibung = []
    for i in ankerpunkte:
        new_object.get_info(export, 'Pos_Artikel_Deutsch_Bezeichnung', liste_artikelbeschreibung, i) 

    #Lieferziel hinzufügen
    liste_lieferziel = []
    for i in ankerpunkte:
        new_object.get_info(export, 'Pos_Artikel_Deutsch_SachmerkmalFeld2', liste_lieferziel, i) 

    #Stückzahl hinzufügen
    liste_stückzahl = []
    for i in ankerpunkte:
        new_object.get_info(export, 'Pos_Menge', liste_stückzahl, i)  
    liste_stückzahl = ['Stückzahl:' + x for x in liste_stückzahl]

    #"Verladen"-String-Liste hinzufügen
    listofverladen = ['Verladen:'] * len(liste_stückzahl)
        
    #Verladetage auflisten (Wochentage)  
    liste_verladetag = new_object.trello_footer_L(export, ankerpunkte, search)
    
    #Indizes der Verladetag-Liste bestimmen
    verladetag_index = [*range(len(liste_verladetag))]
    
    #Indizes der Verladetage, die nur einmal vorkommen
    verladetag_index_unique = [liste_verladetag.index(x) for x in set(liste_verladetag)]
    
    #Anzahl Stapel hinzufügen
    listofstapel = []
    for i in verladetag_index:
        if ((liste_artikelbezeichnung[i] == '0') or (liste_artikelbezeichnung[i] == 'nan')
            or (liste_artikelbezeichnung[i] == liste_artikel[i])):
            anzahl_stapel = -(-int(liste_reine_stückzahl[i])//int(liste_artikel[i]))    #Ergebnis der Division auf die nächste ganze Zahl aufrunden
        else:
            anzahl_stapel = -(-int(liste_reine_stückzahl[i])//int(liste_artikelbezeichnung[i]))

        anzahl_stapel = 'Anzahl Stapel: ' + str(anzahl_stapel)
        listofstapel.append(anzahl_stapel)

    #Aktueller Lagerbestand hinzufügen
    liste_lagerbestand = []
    for i in ankerpunkte:
        liste_lagerbestand = new_object.get_info(export, 'Pos_Artikel_Deutsch_Lagerbestand', liste_lagerbestand, i) 
        #nan-Werte in 0 umwandeln
        liste_lagerbestand = [0 if x == 'nan' else x for x in liste_lagerbestand]
        #alle Werte in Integers umwandeln
        liste_lagerbestand = [int(x) for x in liste_lagerbestand]

    #reine Stückzahlen als Integer auflisten
    liste_reine_stückzahl = []
    for i in ankerpunkte:
        liste_reine_stückzahl = new_object.get_info(export, 'Pos_Menge', liste_reine_stückzahl, i)
        #alle Werte in Integer-Werte umwandeln
        liste_reine_stückzahl = [int(x) for x in liste_reine_stückzahl]

    #Aktueller Auftragsbestand berechnen durch Addition des Lagerbestands und der Stückzahl
    liste_auftragsbestand = [sum(x) for x in zip(liste_lagerbestand, liste_reine_stückzahl)]

    #Test für Beschreibungen zusammenfügen
    liste_beschreibung = []
    
    liste_maschinen = []
    for i in machine_index:
        maschine = liste_maschine[i] + '\n' 
        liste_maschinen.append(maschine)

    liste_beschreibung.append(liste_maschinen)

    liste_verladetermin = []
    for i in machine_index:
        verladetermin = liste_wunschtermin[i][0]
        verladetermin = 'Verladetermin: ' + datetime.strptime(verladetermin, '%Y-%m-%d %H:%M:%S').strftime("%d.%m.%Y")
        liste_verladetermin.append(verladetermin)

    liste_beschreibung.append(liste_verladetermin)
    
    liste_aktuelle_stückzahl = []
    for i in machine_index:
        lagerbestand = "Aktuelle Stückzahl am Lager: " + str(liste_lagerbestand[i])
        liste_aktuelle_stückzahl.append(lagerbestand)

    liste_beschreibung.append(liste_aktuelle_stückzahl)

    liste_aktueller_auftragsbestand = []
    for i in machine_index:
        auftragsbestand = "Aktueller Auftragsbestand: " + str(liste_auftragsbestand[i])
        liste_aktueller_auftragsbestand.append(auftragsbestand)

    liste_beschreibung.append(liste_aktueller_auftragsbestand)

    liste_beschreibung = np.array(liste_beschreibung).T.tolist()
    
    #für jede aktuell im Bestand befindliche Maschine Daten in Karte hochladen
    for i in machine_index: 
        for j in produktionslisten_index:

            if (liste_maschine[i] == alle_produktionslisten[j]):

                P_list_id = liste_P_list_id[j]
                head_P = liste_head_P[i]
                dt = datetime.strptime(str(liste_wunschtermin[i][0]), '%Y-%m-%d %H:%M:%S') #Datumsformat des Fälligkeitsdatums umwandeln 
                dt = dt + timedelta(hours=24) + timedelta(minutes = 59)      #und um 24 Stunden 59 Minuten erhöhen (+eine Stunde wegen Zeitverzug)                                        
                beschreibung = "\n".join(liste_beschreibung[i])       #Beschreibung, die in die Karte geladen werden soll
                name = files[i]                 #Bezeichnung des Anhangs in der Karte
                attachment = files[i]           #Datei, die angehängt werden soll
                stückliste = liste_stückliste[i]             #Daten aus einer Stückliste

                new_object.upload_data_P(liste_auftragsnummerproduktion, 
                                        versandart, 
                                        P_list_id, 
                                        head_P, 
                                        dt,
                                        beschreibung,
                                        name,
                                        attachment,
                                        stückliste_index,
                                        stückliste)
       
    #für jeden im Auftrag auftretenden Verladetag Daten in Karte hochladen, Tabelle in Beschreibung ergänzen, Stücklisten als Anhänge hinzufügen
    #und Checklisten mit Stückzahlen auffüllen
    for i in verladetag_index_unique: 
        for j in logistiklisten_index:

            if (liste_verladetag[i] == alle_logistiklisten[j]):

                maschine = alle_logistiklisten[j]
                L_list_id = liste_L_list_id[j]
                head_L = liste_head_L[i]
                dt = datetime.strptime(str(liste_wunschtermin[i][0]), '%Y-%m-%d %H:%M:%S') #Datumsformat des Fälligkeitsdatums umwandeln 
                dt = dt + timedelta(hours=24) + timedelta(minutes = 59)      #und um 24 Stunden 59 Minuten erhöhen (+eine Stunde wegen Zeitverzug)  
                verladetag = liste_verladetag[i]
                
                new_object.upload_data_L(liste_auftragsnummerproduktion, 
                                        versandart, 
                                        L_list_id, 
                                        head_L, 
                                        dt, 
                                        verladetag,
                                        verladetag_index,
                                        liste_artikelnummer, 
                                        liste_bezugsquelle, 
                                        liste_artikelbeschreibung, 
                                        liste_lieferziel, 
                                        listofstapel, 
                                        liste_stückzahl, 
                                        listofverladen)
    
    #zugehörige Stücklisten in in Ordner 'Erledigt' verschieben, es sei denn sie sind schon dort vorhanden
    dst_dirname = root + '/Erledigt/'
    for s in files:
        try:
            shutil.move(s, dst_dirname)
        except IOError:
            pass
                
#in Trello eingelesene Auftragsdateien in Ordner 'Erledigt' verschieben, es sei denn sie sind schon dort vorhanden
dst_dirname = root + '/Erledigt/'
for file in liste_files:
    try: 
        shutil.move(file, dst_dirname)
    except IOError:
        pass
    
#alle Exportdateien im Ordner 'Erledigt' auflisten
os.chdir(dst_dirname)

liste_tobedone = new_object.get_files(search_for)

#dieselben Dateien im Ursprungsordner löschen
os.chdir(root)
for file in liste_tobedone:
    try:
        os.remove(file)
    except IOError:
        pass